# Sesión 2: Clasificación de Imágenes (Visión Artificial con Keras)
Facilitador: Héctor Martínez | Unidad de Telecomunicaciones - ABAE

En este segundo bloque práctico, daremos el salto desde el modelado de variables físicas unidimensionales hacia el procesamiento de estructuras matriciales complejas (**Tensores**). 

Utilizaremos el benchmark clásico **Fashion-MNIST** para enseñar cómo una red neuronal densa procesa imágenes en escala de grises y aprende a asignar probabilidades discretas a categorías de objetos independientes.

## Objetivos de Aprendizaje:
1. Comprender el preprocesamiento y la normalización escalar de matrices de píxeles.
2. Dominar la capa de aplanado (`Flatten`) para la transformación dimensional de datos.
3. Interpretar la salida probabilística de una capa `Softmax` mediante la función de pérdida *Cross-Entropy*.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

# Carga directa del dataset nativo de Fashion-MNIST
fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Mapeo de nombres de clases según el estándar del dataset
class_names = ['Camiseta', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapato deportivo', 'Bolso', 'Botín']

print(f"Set de entrenamiento: {train_images.shape[0]} imágenes de {train_images.shape[1]}x{train_images.shape[2]} píxeles.")
print(f"Set de prueba:        {test_images.shape[0]} imágenes para auditar el modelo.")

## 1. El Concepto de Tensor, Normalización y Aplanado (Flatten)
Antes de alimentar la red, debemos entender la estructura de entrada. Cada imagen es una matriz de $28 \times 28$ números enteros que representan la intensidad del gris de cada píxel, variando en un rango discreto de 0 (negro absoluto) a 255 (blanco absoluto).

Para asegurar la estabilidad numérica de los optimizadores basados en gradiente, realizaremos una **normalización escalar**, transformando los valores enteros al rango continuo $[0, 1.0]$.

In [ ]:
# Inspección visual de la primera muestra del set de datos
plt.figure(figsize=(5,5))
plt.imshow(train_images[0], cmap=plt.cm.binary)
plt.title(f"Clase Real: {class_names[train_labels[0]]}")
plt.colorbar()
plt.grid(False)
plt.show()

# --- OPERACIÓN DE NORMALIZACIÓN ESCALAR ---
train_images = train_images / 255.0
test_images = test_images / 255.0
print("--> Datos normalizados exitosamente en el rango [0.0, 1.0]")

## 2. Construcción de la Red de Clasificación Multiclase
A diferencia de la Sesión 1, donde nuestra capa de salida tenía un único nodo continuo, aquí requerimos **10 neuronas en la capa de salida** (una por cada categoría). 

La mecánica de la red se dividirá estrictamente en dos etapas: la definición estructural del flujo de tensores y la configuración analítica de su optimización.

In [ ]:
# =====================================================================
# ETAPA A: DEFINICIÓN ARQUITECTÓNICA DE LA RED (FLUJO DE TENSORES)
# =====================================================================
model_clasificador = keras.Sequential([
    # 1. Capa Flatten: Desenrola la matriz de 28x28 en un vector plano de 784 entradas
    keras.layers.Flatten(input_shape=(28, 28)),
    
    # 2. Capa Oculta Densa: Extrae características abstractas del vector de píxeles
    keras.layers.Dense(units=128, activation='relu'),
    
    # 3. Capa de Salida Softmax: Calcula una distribución de probabilidad para las 10 clases
    keras.layers.Dense(units=10, activation='softmax')
])

print("Resumen de la arquitectura diseñada:")
model_clasificador.summary()

In [ ]:
# =====================================================================
# ETAPA B: PREPARACIÓN PARA LA COMPILACIÓN (MÉTRICAS Y PÉRDIDA)
# =====================================================================
model_clasificador.compile(
    optimizer='adam',
    # Usamos Sparse Categorical Crossentropy porque las etiquetas son números enteros (0 a 9)
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'] # Tasa de acierto directo para monitoreo humano
)

# Ejecución del entrenamiento en el lote de datos completo
print("Iniciando el entrenamiento del clasificador multiclase...")
history_clasificador = model_clasificador.fit(train_images, train_labels, epochs=10, verbose=1)
print("\n¡Entrenamiento completado!")

In [ ]:
# Validación del modelo frente a datos que nunca observó durante el entrenamiento
test_loss, test_acc = model_clasificador.evaluate(test_images, test_labels, verbose=2)
print(f"\n--> Precisión General de Testeo (Accuracy): {test_acc*100:.2f}%")

## 3. Reto Autónomo: Auditoría de Errores e Interpretación de Probabilidades
**Propósito del Reto:** Desarrollar un criterio analítico para evaluar modelos de IA en producción. Un ingeniero analista no debe conformarse con una métrica global de precisión (*Accuracy*); debe auditar en qué escenarios específicos se equivoca el algoritmo.

Al invocar el método `model.predict()`, la red no devuelve una etiqueta fija, sino un **vector de 10 probabilidades independientes** calculadas por la función `Softmax`. La clase elegida es aquella que posea el valor máximo del vector (el índice con mayor nivel de confianza).

### Instrucciones del Reto Técnico:
1. Ejecuta la celda de abajo para calcular las predicciones probabilísticas de todo el conjunto de prueba.
2. Desarrolla una pequeña rutina en Python que compare el índice de la predicción más alta (`np.argmax()`) con la etiqueta real (`test_labels`).
3. Filtra, identifica y **despliega visualmente en pantalla tres (3) imágenes específicas donde el modelo haya fallado de forma contundente** (por ejemplo, casos donde la red asignó una probabilidad muy alta a una categoría incorrecta).
4. Añade una celda de texto final con tus conclusiones: ¿El error de la IA es comprensible por ambigüedad visual compartida en los píxeles (ej. confundir Camisa con Suéter) o representa un fallo de generalización estructural?

In [ ]:
# Generación del vector de salida probabilístico para todas las muestras de prueba
predicciones_probabilísticas = model_clasificador.predict(test_images)

# Ejemplo de inspección de la primera muestra de prueba:
print("Estructura del vector Softmax para la muestra [0]:\n", predicciones_probabilísticas[0])
print(f"\n--> Índice con mayor probabilidad calculada: {np.argmax(predicciones_probabilísticas[0])}")
print(f"--> Etiqueta real registrada en el dataset:   {test_labels[0]}")

print("\n====================================================================")
print("¡RETO ACTIVADO! Desarrolla tu lógica de filtrado y visualización abajo.")
print("====================================================================")

# TODO: Escribe aquí tu código para identificar y graficar las 3 imágenes fallidas.
# Pista: Puedes usar un ciclo 'for' combinando plt.subplot() para mostrar las imágenes y sus errores.